# Phase 2: Building the Heart Disease Risk Model
This notebook covers loading the dataset, preprocessing, training an XGBoost model, explaining it with SHAP, and saving it for our FastAPI backend.

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report, confusion_matrix
import xgboost as xgb
import shap
import joblib
import os

import warnings
warnings.filterwarnings('ignore')

# Ensure models directory exists
os.makedirs('../../backend/models', exist_ok=True)

## Step 2: Load the Data
We load the clean Cleveland Heart Disease dataset.

In [ ]:
df = pd.read_csv('../data/heart_disease.csv')
df.head()

## Step 3: Data Preprocessing
X is our inputs, y is our target (disease). We split 80/20 for training and testing. Then we scale the data using StandardScaler.

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to pandas DataFrames with column names for SHAP later
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)
print(f'Training Data Shape: {X_train_scaled.shape}')

## Step 4: Train the XGBoost Model

In [ ]:
model = xgb.XGBClassifier(
    objective='binary:logistic', 
    eval_metric='logloss',
    use_label_encoder=False,
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print('Recall (Most Important metric):', recall_score(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred))

## Step 5: Save (Serialize) Model and Scaler

In [ ]:
# We save these inside the backend/models folder directly!
joblib.dump(model, '../../backend/models/xgboost_model.pkl')
joblib.dump(scaler, '../../backend/models/scaler.pkl')

# Create and save a TreeExplainer for SHAP
explainer = shap.TreeExplainer(model)
joblib.dump(explainer, '../../backend/models/shap_explainer.pkl')

print('Model, Scaler, and Explainer have been successfully saved to the backend API directory!')